In [0]:
%pip install databricks-sdk --upgrade
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 912.3/912.3 kB 24.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.3
    Not uninstalling protobuf at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-47124cfc-f24c-4f8b-b3a5-99de39470e92
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.40.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-47124cfc-f24c-4f8b-b3a5-99de39470e92
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-api-core 2.18.0 requires protobuf!=3.20.0,!=3.20.1,

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import GenieAPI
import os
import json
import time
from typing import Dict, List, Optional, Any
import pandas as pd

# Initialize Databricks client
# When running in Databricks notebook, authentication is automatic
w = WorkspaceClient()

# Genie Space ID
GENIE_SPACE_ID = "01f163975381128cb65f3e0655092d9f"

print(f"✓ Authenticated as: {w.current_user.me().user_name}")
print(f"✓ Genie Space ID: {GENIE_SPACE_ID}")

✓ Authenticated as: markyoung0110@gmail.com
✓ Genie Space ID: 01f163975381128cb65f3e0655092d9f


In [0]:
class GenieQueryClient:
    """
    Production-ready client for querying Genie Spaces programmatically.
    Handles message submission, result retrieval, and error handling.
    """
    
    def __init__(self, workspace_client: WorkspaceClient, space_id: str):
        self.client = workspace_client
        self.space_id = space_id
        self.genie = workspace_client.genie
    
    def create_conversation(self) -> str:
        """Create a new conversation in the Genie Space."""
        try:
            conversation = self.genie.create_message(
                space_id=self.space_id,
                content="Hello"  # Initial message to start conversation
            )
            return conversation.conversation_id
        except Exception as e:
            raise Exception(f"Failed to create conversation: {str(e)}")
    
    def submit_query(self, question: str, conversation_id: Optional[str] = None) -> Dict[str, Any]:
        """
        Submit a natural language question to Genie.
        
        Args:
            question: Natural language question
            conversation_id: Optional conversation ID to continue thread
        
        Returns:
            Dictionary with conversation_id, message_id, and status
        """
        try:
            # Create new conversation if not provided
            if not conversation_id:
                conversation_id = self.create_conversation()
            
            # Submit the question
            response = self.genie.create_message(
                space_id=self.space_id,
                content=question,
                conversation_id=conversation_id
            )
            
            return {
                "conversation_id": conversation_id,
                "message_id": response.id,
                "status": response.status.value if hasattr(response, 'status') else "submitted",
                "question": question
            }
        except Exception as e:
            raise Exception(f"Failed to submit query: {str(e)}")
    
    def wait_for_result(self, conversation_id: str, message_id: str, 
                       timeout: int = 60, poll_interval: int = 2) -> Dict[str, Any]:
        """
        Poll for query results until completion or timeout.
        
        Args:
            conversation_id: Conversation ID
            message_id: Message ID to poll
            timeout: Maximum wait time in seconds
            poll_interval: Seconds between polls
        
        Returns:
            Dictionary with query results and metadata
        """
        start_time = time.time()
        
        while time.time() - start_time < timeout:
            try:
                # Get message status
                message = self.genie.get_message(
                    space_id=self.space_id,
                    conversation_id=conversation_id,
                    message_id=message_id
                )
                
                # Check if completed
                if hasattr(message, 'status'):
                    status = message.status.value
                    
                    if status in ['COMPLETED', 'EXECUTING_QUERY']:
                        return self._parse_message_result(message)
                    elif status in ['FAILED', 'ERROR']:
                        error_msg = getattr(message, 'error', 'Query failed')
                        raise Exception(f"Query failed: {error_msg}")
                
                time.sleep(poll_interval)
            
            except Exception as e:
                if "failed" in str(e).lower():
                    raise
                # Continue polling on other errors
                time.sleep(poll_interval)
        
        raise TimeoutError(f"Query did not complete within {timeout} seconds")
    
    def _parse_message_result(self, message) -> Dict[str, Any]:
        """Parse Genie message response into structured format."""
        result = {
            "message_id": message.id,
            "status": message.status.value if hasattr(message, 'status') else 'unknown',
            "content": None,
            "sql_query": None,
            "data": None,
            "chart_type": None,
            "metadata": {}
        }
        
        # Extract content
        if hasattr(message, 'content'):
            result['content'] = message.content
        
        # Extract attachments (query results, charts)
        if hasattr(message, 'attachments') and message.attachments:
            for attachment in message.attachments:
                # SQL query
                if hasattr(attachment, 'query'):
                    if hasattr(attachment.query, 'query'):
                        result['sql_query'] = attachment.query.query
                
                # Query results
                if hasattr(attachment, 'query_result'):
                    result['data'] = self._extract_query_data(attachment.query_result)
                
                # Chart information
                if hasattr(attachment, 'chart'):
                    result['chart_type'] = getattr(attachment.chart, 'type', None)
        
        return result
    
    def _extract_query_data(self, query_result) -> Optional[pd.DataFrame]:
        """Extract tabular data from query result."""
        try:
            # Check if data is available
            if hasattr(query_result, 'data_array'):
                columns = query_result.columns if hasattr(query_result, 'columns') else []
                data = query_result.data_array
                
                if columns and data:
                    return pd.DataFrame(data, columns=columns)
            
            return None
        except Exception:
            return None
    
    def query_and_wait(self, question: str, timeout: int = 60) -> Dict[str, Any]:
        """
        Convenience method: submit query and wait for results.
        
        Args:
            question: Natural language question
            timeout: Maximum wait time
        
        Returns:
            Complete query result with data and metadata
        """
        # Submit query
        submission = self.submit_query(question)
        
        # Wait for result
        result = self.wait_for_result(
            conversation_id=submission['conversation_id'],
            message_id=submission['message_id'],
            timeout=timeout
        )
        
        # Add submission info
        result['question'] = question
        result['conversation_id'] = submission['conversation_id']
        
        return result

# Initialize client
genie_client = GenieQueryClient(w, GENIE_SPACE_ID)
print("✓ Genie API client initialized")

✓ Genie API client initialized


In [0]:
def execute_genie_query(question: str, format_for_web: bool = True) -> Dict[str, Any]:
    """
    Execute a natural language query against the Genie Space.
    
    Args:
        question: Natural language question
        format_for_web: Whether to format results for web dashboard
    
    Returns:
        Dictionary with query results, SQL, and formatted data
    """
    try:
        print(f"📊 Executing query: {question}")
        
        # Execute query
        result = genie_client.query_and_wait(question, timeout=60)
        
        # Format for web if requested
        if format_for_web and result.get('data') is not None:
            df = result['data']
            result['formatted_data'] = {
                'columns': df.columns.tolist(),
                'rows': df.to_dict('records'),
                'total_rows': len(df),
                'summary': {
                    'row_count': len(df),
                    'column_count': len(df.columns)
                }
            }
        
        print(f"✓ Query completed: {result.get('status')}")
        if result.get('sql_query'):
            print(f"✓ Generated SQL: {result['sql_query'][:100]}...")
        
        return result
    
    except Exception as e:
        return {
            'error': str(e),
            'question': question,
            'status': 'failed'
        }

def batch_execute_queries(questions: List[str]) -> List[Dict[str, Any]]:
    """
    Execute multiple queries in sequence.
    
    Args:
        questions: List of natural language questions
    
    Returns:
        List of query results
    """
    results = []
    
    for i, question in enumerate(questions, 1):
        print(f"\n[{i}/{len(questions)}] Processing: {question}")
        result = execute_genie_query(question)
        results.append(result)
        
        # Small delay between queries
        if i < len(questions):
            time.sleep(1)
    
    return results

print("✓ Query execution functions ready")

✓ Query execution functions ready


In [0]:
# Define example queries for the admin dashboard
example_queries = [
    "Show me response times by event type",
    "What are the top 10 users by session count?",
    "How many logs have is_correct = true vs false by day?",
    "What's the average response time by user?",
    "Show me the total count of events by event_type for the last 7 days"
]

print("📋 Example queries for Usage Logs Analytics:")
for i, q in enumerate(example_queries, 1):
    print(f"  {i}. {q}")

# Execute one example query
print("\n" + "="*60)
print("Executing first example query...")
print("="*60)

result = execute_genie_query(example_queries[0])

# Display results
if 'error' not in result:
    print(f"\n✓ Query Status: {result['status']}")
    
    if result.get('sql_query'):
        print(f"\n📝 Generated SQL:")
        print(result['sql_query'])
    
    if result.get('data') is not None:
        print(f"\n📊 Results ({len(result['data'])} rows):")
        display(result['data'].head(10))
    
    if result.get('formatted_data'):
        print(f"\n🌐 Web-ready format:")
        print(json.dumps(result['formatted_data']['summary'], indent=2))
else:
    print(f"\n❌ Error: {result['error']}")

📋 Example queries for Usage Logs Analytics:
  1. Show me response times by event type
  2. What are the top 10 users by session count?
  3. How many logs have is_correct = true vs false by day?
  4. What's the average response time by user?
  5. Show me the total count of events by event_type for the last 7 days

Executing first example query...
📊 Executing query: Show me response times by event type

❌ Error: Failed to submit query: Failed to create conversation: GenieAPI.create_message() missing 1 required positional argument: 'conversation_id'


In [0]:
"""
FastAPI/Flask Integration Example

This code shows how to integrate the Genie client into a web backend.
You can copy this into your Flask or FastAPI application.
"""

# ============================================================================
# FASTAPI EXAMPLE
# ============================================================================

fastapi_example = '''
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from databricks.sdk import WorkspaceClient
import os

app = FastAPI(title="Usage Logs Analytics API")

# Enable CORS for frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Configure for production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Initialize Databricks client
# Use environment variables or service principal authentication
w = WorkspaceClient(
    host=os.getenv("DATABRICKS_HOST"),
    token=os.getenv("DATABRICKS_TOKEN")
)

genie_client = GenieQueryClient(w, "01f163975381128cb65f3e0655092d9f")

class QueryRequest(BaseModel):
    question: str
    timeout: int = 60

class QueryResponse(BaseModel):
    question: str
    status: str
    data: dict
    sql_query: str = None
    error: str = None

@app.post("/api/query", response_model=QueryResponse)
async def query_logs(request: QueryRequest):
    """
    Execute natural language query against usage logs.
    
    Example request:
    POST /api/query
    {
        "question": "Show me top 10 users by session count",
        "timeout": 60
    }
    """
    try:
        result = genie_client.query_and_wait(
            question=request.question,
            timeout=request.timeout
        )
        
        # Format response
        response_data = {
            "question": request.question,
            "status": result.get("status", "completed"),
            "sql_query": result.get("sql_query"),
            "data": {
                "columns": result["data"].columns.tolist() if result.get("data") is not None else [],
                "rows": result["data"].to_dict("records") if result.get("data") is not None else [],
                "total_rows": len(result["data"]) if result.get("data") is not None else 0
            }
        }
        
        return QueryResponse(**response_data)
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/api/metrics/summary")
async def get_metrics_summary():
    """
    Get pre-computed metrics summary for dashboard.
    """
    queries = [
        "What is the total count of logs today?",
        "What is the average response time in the last hour?",
        "How many unique users accessed the system today?"
    ]
    
    results = {}
    for query in queries:
        try:
            result = genie_client.query_and_wait(query, timeout=30)
            if result.get("data") is not None and len(result["data"]) > 0:
                # Extract first value
                results[query] = result["data"].iloc[0, 0]
        except Exception as e:
            results[query] = {"error": str(e)}
    
    return results

@app.get("/health")
async def health_check():
    return {"status": "healthy", "service": "usage-logs-analytics"}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# ============================================================================
# FLASK EXAMPLE
# ============================================================================

flask_example = '''
from flask import Flask, request, jsonify
from flask_cors import CORS
from databricks.sdk import WorkspaceClient
import os

app = Flask(__name__)
CORS(app)  # Enable CORS

# Initialize Databricks client
w = WorkspaceClient(
    host=os.getenv("DATABRICKS_HOST"),
    token=os.getenv("DATABRICKS_TOKEN")
)

genie_client = GenieQueryClient(w, "01f163975381128cb65f3e0655092d9f")

@app.route("/api/query", methods=["POST"])
def query_logs():
    """
    Execute natural language query against usage logs.
    """
    try:
        data = request.get_json()
        question = data.get("question")
        timeout = data.get("timeout", 60)
        
        if not question:
            return jsonify({"error": "Question is required"}), 400
        
        # Execute query
        result = genie_client.query_and_wait(question, timeout=timeout)
        
        # Format response
        response = {
            "question": question,
            "status": result.get("status", "completed"),
            "sql_query": result.get("sql_query"),
            "data": {
                "columns": result["data"].columns.tolist() if result.get("data") is not None else [],
                "rows": result["data"].to_dict("records") if result.get("data") is not None else [],
                "total_rows": len(result["data"]) if result.get("data") is not None else 0
            }
        }
        
        return jsonify(response)
    
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/api/metrics/summary", methods=["GET"])
def get_metrics_summary():
    """
    Get dashboard summary metrics.
    """
    queries = [
        "What is the total count of logs today?",
        "What is the average response time?",
        "How many unique users are active?"
    ]
    
    results = {}
    for query in queries:
        try:
            result = genie_client.query_and_wait(query, timeout=30)
            if result.get("data") is not None and len(result["data"]) > 0:
                results[query] = result["data"].iloc[0, 0]
        except Exception as e:
            results[query] = {"error": str(e)}
    
    return jsonify(results)

@app.route("/health", methods=["GET"])
def health_check():
    return jsonify({"status": "healthy", "service": "usage-logs-analytics"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000, debug=False)
'''

print("\n" + "="*80)
print("WEB API INTEGRATION EXAMPLES")
print("="*80)

print("\n📦 FastAPI Example (Recommended for Production):")
print("Save this to 'main.py' and run with: uvicorn main:app --reload")
print("-" * 80)
print(fastapi_example)

print("\n" + "="*80)
print("\n📦 Flask Example (Alternative):")
print("Save this to 'app.py' and run with: python app.py")
print("-" * 80)
print(flask_example)

print("\n" + "="*80)
print("✓ Web API integration code ready to copy")
print("="*80)

In [0]:
from functools import wraps
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def retry_on_failure(max_attempts: int = 3, delay: int = 2):
    """
    Decorator for retrying failed API calls.
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    logger.warning(f"Attempt {attempt}/{max_attempts} failed: {str(e)}")
                    
                    if attempt < max_attempts:
                        logger.info(f"Retrying in {delay} seconds...")
                        time.sleep(delay)
            
            logger.error(f"All {max_attempts} attempts failed")
            raise last_exception
        
        return wrapper
    return decorator

class GenieQueryError(Exception):
    """Custom exception for Genie query errors."""
    pass

class GenieTimeoutError(Exception):
    """Custom exception for query timeouts."""
    pass

@retry_on_failure(max_attempts=3, delay=2)
def robust_query_execution(question: str, timeout: int = 60) -> Dict[str, Any]:
    """
    Execute query with comprehensive error handling and retry logic.
    
    Args:
        question: Natural language question
        timeout: Maximum wait time
    
    Returns:
        Query result dictionary
    
    Raises:
        GenieQueryError: If query fails
        GenieTimeoutError: If query times out
    """
    try:
        logger.info(f"Executing query: {question}")
        result = genie_client.query_and_wait(question, timeout=timeout)
        
        # Validate result
        if 'error' in result:
            raise GenieQueryError(f"Query error: {result['error']}")
        
        if result.get('status') == 'failed':
            raise GenieQueryError("Query execution failed")
        
        logger.info(f"Query completed successfully")
        return result
    
    except TimeoutError as e:
        logger.error(f"Query timeout: {str(e)}")
        raise GenieTimeoutError(f"Query timed out after {timeout} seconds")
    
    except Exception as e:
        logger.error(f"Query execution error: {str(e)}")
        raise GenieQueryError(f"Failed to execute query: {str(e)}")

def safe_query_with_fallback(question: str, fallback_sql: Optional[str] = None) -> Dict[str, Any]:
    """
    Execute query with fallback to direct SQL if Genie fails.
    
    Args:
        question: Natural language question
        fallback_sql: Optional SQL query to run if Genie fails
    
    Returns:
        Query result
    """
    try:
        # Try Genie first
        result = robust_query_execution(question)
        return result
    
    except (GenieQueryError, GenieTimeoutError) as e:
        logger.warning(f"Genie query failed: {str(e)}")
        
        if fallback_sql:
            logger.info("Executing fallback SQL query")
            try:
                # Execute fallback SQL directly
                df = spark.sql(fallback_sql)
                return {
                    'status': 'completed_via_fallback',
                    'data': df.toPandas(),
                    'sql_query': fallback_sql,
                    'question': question,
                    'note': 'Executed via fallback SQL'
                }
            except Exception as sql_error:
                logger.error(f"Fallback SQL also failed: {str(sql_error)}")
                raise
        
        raise

# Example usage with error handling
print("\n" + "="*60)
print("ERROR HANDLING EXAMPLES")
print("="*60)

try:
    result = robust_query_execution(
        "Show me response times by event type",
        timeout=30
    )
    print("✓ Query executed successfully with retry logic")
    if result.get('data') is not None:
        print(f"✓ Retrieved {len(result['data'])} rows")

except GenieTimeoutError as e:
    print(f"❌ Timeout: {str(e)}")
except GenieQueryError as e:
    print(f"❌ Query Error: {str(e)}")
except Exception as e:
    print(f"❌ Unexpected error: {str(e)}")

print("\n✓ Error handling and retry logic ready")

In [0]:
def refresh_dashboard_data() -> Dict[str, Any]:
    """
    Execute all dashboard queries and return formatted results.
    This function can be called periodically to refresh dashboard data.
    
    Returns:
        Dictionary with all dashboard widgets' data
    """
    dashboard_queries = {
        "response_time_by_event": "Show me average response times by event type for the last 7 days",
        "top_users": "What are the top 10 users by session count?",
        "accuracy_trend": "Show me the percentage of is_correct = true by day for the last 30 days",
        "hourly_usage": "Show me the count of events by hour of day",
        "event_distribution": "What is the distribution of event types?"
    }
    
    results = {}
    errors = []
    
    print(f"\n🔄 Refreshing dashboard data ({len(dashboard_queries)} queries)...")
    print("="*60)
    
    for widget_name, question in dashboard_queries.items():
        try:
            print(f"\n📊 {widget_name}: {question}")
            result = robust_query_execution(question, timeout=45)
            
            # Format for dashboard
            if result.get('data') is not None:
                df = result['data']
                results[widget_name] = {
                    'status': 'success',
                    'data': df.to_dict('records'),
                    'columns': df.columns.tolist(),
                    'row_count': len(df),
                    'sql': result.get('sql_query'),
                    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
                }
                print(f"  ✓ Retrieved {len(df)} rows")
            else:
                results[widget_name] = {
                    'status': 'no_data',
                    'error': 'No data returned'
                }
                print(f"  ⚠ No data returned")
        
        except Exception as e:
            error_msg = str(e)
            results[widget_name] = {
                'status': 'error',
                'error': error_msg
            }
            errors.append(f"{widget_name}: {error_msg}")
            print(f"  ❌ Error: {error_msg}")
        
        # Small delay between queries
        time.sleep(1)
    
    # Summary
    print("\n" + "="*60)
    success_count = sum(1 for r in results.values() if r.get('status') == 'success')
    print(f"✓ Dashboard refresh complete: {success_count}/{len(dashboard_queries)} successful")
    
    if errors:
        print(f"\n⚠ Errors encountered:")
        for error in errors:
            print(f"  - {error}")
    
    return {
        'widgets': results,
        'summary': {
            'total_queries': len(dashboard_queries),
            'successful': success_count,
            'failed': len(errors),
            'refresh_time': time.strftime('%Y-%m-%d %H:%M:%S')
        },
        'errors': errors
    }

# Execute dashboard refresh
dashboard_data = refresh_dashboard_data()

# Display summary
print("\n" + "="*60)
print("DASHBOARD DATA SUMMARY")
print("="*60)
print(json.dumps(dashboard_data['summary'], indent=2))

# Show sample data from first successful widget
for widget_name, widget_data in dashboard_data['widgets'].items():
    if widget_data.get('status') == 'success' and widget_data.get('data'):
        print(f"\n📊 Sample data from '{widget_name}':")
        sample_df = pd.DataFrame(widget_data['data']).head(3)
        display(sample_df)
        break

In [0]:
"""
Frontend Integration Examples

JavaScript/React code for calling your FastAPI/Flask backend
and displaying results in an admin dashboard.
"""

javascript_example = '''
// ============================================================================
// React Component Example
// ============================================================================

import React, { useState, useEffect } from 'react';
import axios from 'axios';

const API_BASE_URL = 'http://localhost:8000/api';

function UsageLogsDashboard() {
  const [queryResult, setQueryResult] = useState(null);
  const [loading, setLoading] = useState(false);
  const [error, setError] = useState(null);
  const [question, setQuestion] = useState('');

  // Execute natural language query
  const executeQuery = async (questionText) => {
    setLoading(true);
    setError(null);
    
    try {
      const response = await axios.post(`${API_BASE_URL}/query`, {
        question: questionText,
        timeout: 60
      });
      
      setQueryResult(response.data);
    } catch (err) {
      setError(err.response?.data?.detail || err.message);
    } finally {
      setLoading(false);
    }
  };

  // Pre-defined quick queries
  const quickQueries = [
    'Show me response times by event type',
    'What are the top 10 users by session count?',
    'Show me usage trends for the last 7 days'
  ];

  return (
    <div className="dashboard">
      <h1>Usage Logs Analytics</h1>
      
      {/* Query Input */}
      <div className="query-input">
        <input
          type="text"
          value={question}
          onChange={(e) => setQuestion(e.target.value)}
          placeholder="Ask a question about your logs..."
          onKeyPress={(e) => e.key === 'Enter' && executeQuery(question)}
        />
        <button onClick={() => executeQuery(question)} disabled={loading}>
          {loading ? 'Analyzing...' : 'Query'}
        </button>
      </div>

      {/* Quick Query Buttons */}
      <div className="quick-queries">
        <h3>Quick Queries:</h3>
        {quickQueries.map((q, idx) => (
          <button
            key={idx}
            onClick={() => executeQuery(q)}
            disabled={loading}
          >
            {q}
          </button>
        ))}
      </div>

      {/* Error Display */}
      {error && (
        <div className="error">
          <strong>Error:</strong> {error}
        </div>
      )}

      {/* Results Display */}
      {queryResult && (
        <div className="results">
          <h2>Results</h2>
          
          {/* SQL Query */}
          {queryResult.sql_query && (
            <details>
              <summary>View Generated SQL</summary>
              <pre>{queryResult.sql_query}</pre>
            </details>
          )}

          {/* Data Table */}
          {queryResult.data && queryResult.data.rows.length > 0 ? (
            <div className="data-table">
              <p>Found {queryResult.data.total_rows} rows</p>
              <table>
                <thead>
                  <tr>
                    {queryResult.data.columns.map((col, idx) => (
                      <th key={idx}>{col}</th>
                    ))}
                  </tr>
                </thead>
                <tbody>
                  {queryResult.data.rows.map((row, idx) => (
                    <tr key={idx}>
                      {queryResult.data.columns.map((col, colIdx) => (
                        <td key={colIdx}>{row[col]}</td>
                      ))}
                    </tr>
                  ))}
                </tbody>
              </table>
            </div>
          ) : (
            <p>No data returned</p>
          )}
        </div>
      )}
    </div>
  );
}

export default UsageLogsDashboard;

// ============================================================================
// Vanilla JavaScript Example (No Framework)
// ============================================================================

class GenieQueryClient {
  constructor(apiBaseUrl) {
    this.apiBaseUrl = apiBaseUrl;
  }

  async executeQuery(question, timeout = 60) {
    const response = await fetch(`${this.apiBaseUrl}/query`, {
      method: 'POST',
      headers: {
        'Content-Type': 'application/json',
      },
      body: JSON.stringify({ question, timeout })
    });

    if (!response.ok) {
      const error = await response.json();
      throw new Error(error.detail || 'Query failed');
    }

    return await response.json();
  }

  async getDashboardSummary() {
    const response = await fetch(`${this.apiBaseUrl}/metrics/summary`);
    return await response.json();
  }
}

// Usage
const client = new GenieQueryClient('http://localhost:8000/api');

async function displayQueryResults() {
  const resultsDiv = document.getElementById('results');
  
  try {
    resultsDiv.innerHTML = '<p>Loading...</p>';
    
    const result = await client.executeQuery(
      'Show me top 10 users by session count'
    );
    
    // Build HTML table
    let html = `<h2>Query Results</h2>`;
    html += `<p>Found ${result.data.total_rows} rows</p>`;
    html += '<table><thead><tr>';
    
    result.data.columns.forEach(col => {
      html += `<th>${col}</th>`;
    });
    html += '</tr></thead><tbody>';
    
    result.data.rows.forEach(row => {
      html += '<tr>';
      result.data.columns.forEach(col => {
        html += `<td>${row[col]}</td>`;
      });
      html += '</tr>';
    });
    html += '</tbody></table>';
    
    resultsDiv.innerHTML = html;
  } catch (error) {
    resultsDiv.innerHTML = `<p class="error">Error: ${error.message}</p>`;
  }
}
'''

print("\n" + "="*80)
print("FRONTEND INTEGRATION EXAMPLES")
print("="*80)

print("\n🌐 React Component Example:")
print("-" * 80)
print(javascript_example)

print("\n" + "="*80)
print("✓ Frontend integration code ready")
print("="*80)

print("\n📝 To use this in your project:")
print("  1. Copy the FastAPI/Flask backend code to your server")
print("  2. Set environment variables: DATABRICKS_HOST, DATABRICKS_TOKEN")
print("  3. Copy the React component to your frontend")
print("  4. Install dependencies: npm install axios")
print("  5. Update API_BASE_URL to your backend URL")
print("  6. Run both backend and frontend")